In [1]:
import mailbox
import numpy as np
from pathlib import Path

from usenet_no.mbox_utils import message_factory, get_message_body

embedding_dir = Path("../data/embeddings/codefuse-ai/F2LLM-v2-0.6B")
source_dirs = {
    "ia": Path("../data/internet_archive/utf_8_data"),
    "nwa": Path("../data/nwa_90s/utf_8_data"),
}

all_embeddings = []
embedding_indexer = []
text_indexer = []

for f in sorted(embedding_dir.iterdir()):
    if f.stem.endswith("_index"):
        continue
    embs = np.load(f)
    if len(embs) < 400 or len(embs) > 600:
        continue

    mbox_stem, source = f.stem.rsplit("_", 1)
    mbox_file = source_dirs[source] / f"{mbox_stem}.mbox"
    messages = list(mailbox.mbox(str(mbox_file), factory=message_factory))

    index_file = embedding_dir / f"{f.stem}_index.npy"
    if index_file.exists():
        indices = np.load(index_file)
        bodies = [get_message_body(messages[i]) for i in indices]
    else:
        bodies = [get_message_body(m) for m in messages]

    all_embeddings.extend(embs)
    embedding_indexer += [f.stem] * len(embs)
    text_indexer += bodies

len(embedding_indexer), len(all_embeddings), len(text_indexer)

(10066, 10066, 10066)

In [2]:
from collections import Counter

# We will use newsgroups and data source info in plot below
newsgroups_indexer = [file_stem.split("_")[0] for file_stem in embedding_indexer]
sources_indexer = [file_stem.split("_")[1] for file_stem in embedding_indexer]

print(f"Number of messages per source: {Counter(sources_indexer)}")
print(f"Number of messages per newsgroup: {Counter(newsgroups_indexer)}")

Number of messages per source: Counter({'nwa': 5098, 'ia': 4968})
Number of messages per newsgroup: Counter({'no.lisp': 983, 'no.mail.drift': 577, 'no.news.drift': 572, 'no.skole.diverse': 568, 'no.alt.marked.seksualitet': 550, 'no.folklore.overtro': 532, 'no.it.diverse': 527, 'no.video': 511, 'no.annonser.it.unix': 500, 'no.fag.sjukepleie': 500, 'no.kultur.folklore.diverse': 500, 'no.org.efn.diskusjon': 494, 'no.alt.irctreff': 493, 'no.psykolog': 486, 'no.hobby.diverse': 484, 'no.slekt.programmer': 459, 'no.annonser.it.nettverk': 450, 'no.uninett.diverse': 444, 'no.ai': 436})


In [3]:
all_embeddings = np.array(all_embeddings)
all_embeddings.shape

(10066, 1024)

## Dimensionality reduction for plotting with UMAP
The UMAP algorithm reduces the high-dimensional document embeddings (1024 dimensions) to 2 dimensions, so we can plot them as x,y coordinates 

In [4]:
import umap

umap_2d_embeddings = umap.UMAP().fit_transform(all_embeddings)
umap_2d_embeddings.shape

(10066, 2)

In [ ]:
import colorsys
import plotly.graph_objects as go


def hsl_to_hex(h, s, lightness):
    r, g, b = colorsys.hls_to_rgb(h / 360, lightness / 100, s / 100)
    return f"#{int(r * 255):02x}{int(g * 255):02x}{int(b * 255):02x}"


symbol_map = {"nwa": "circle", "ia": "triangle-up"}
unique_newsgroups = sorted(set(newsgroups_indexer))
color_map = {
    ng: hsl_to_hex(int(i * 360 / len(unique_newsgroups)), 70, 50)
    for i, ng in enumerate(unique_newsgroups)
}

hover_texts = np.array(
    [
        f"<b>{stem}</b><br>" + body[:400].replace("\n", "<br>")
        for stem, body in zip(embedding_indexer, text_indexer)
    ]
)

fig = go.Figure()

for ng in unique_newsgroups:
    for source, symbol in symbol_map.items():
        mask = np.array(
            [
                s == source and n == ng
                for s, n in zip(sources_indexer, newsgroups_indexer)
            ]
        )
        if not mask.any():
            continue
        fig.add_trace(
            go.Scattergl(
                x=umap_2d_embeddings[mask, 0],
                y=umap_2d_embeddings[mask, 1],
                mode="markers",
                marker=dict(size=6, color=color_map[ng], symbol=symbol, opacity=0.7),
                name=f"{ng} ({source})",
                text=hover_texts[mask],
                hovertemplate="%{text}<extra></extra>",
            )
        )

fig.update_layout(
    title="Norwegian Usenet message embeddings (color=newsgroup, shape=source)",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    width=1000,
    height=700,
    legend=dict(font=dict(size=9)),
)
fig.show()